# BML — Animated Grid Evolution

Watch the BML grid update in real time across three density regimes:

| Cell | Density | Expected behaviour |
|---|---|---|
| Free flow | $\rho = 0.10$ | Cars move freely; few clusters |
| Critical | $\rho \approx \rho_c$ | Self-organised diagonal stripes |
| Gridlock | $\rho = 0.50$ | Grid freezes permanently |

**How to play:** run all cells — each produces an inline HTML5 animation.
Use the play/pause button below the animation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.animation as animation
from matplotlib.patches import Patch
from IPython.display import HTML

plt.rcParams.update({'figure.dpi': 100})

L        = 80    # grid size — smaller = faster animation
T_WARMUP = 200   # steps before recording starts
T_FRAMES = 120   # animation frames
INTERVAL = 80    # ms between frames


class BML:
    def __init__(self, L, density, r_horiz=0.5, seed=None):
        self.L = L
        rng    = np.random.default_rng(seed)
        n_total = int(round(density * L * L))
        n_horiz = int(round(r_horiz * n_total))
        n_vert  = n_total - n_horiz
        flat      = rng.permutation(L * L)
        self.grid = np.zeros(L * L, dtype=np.int8)
        self.grid[flat[:n_horiz]]                 = 1
        self.grid[flat[n_horiz:n_horiz + n_vert]] = 2
        self.grid = self.grid.reshape(L, L)

    def step(self):
        for axis, car_type, roll_chk, roll_dst, val in [
            (1, 1, -1,  1, 1),
            (0, 2,  1, -1, 2),
        ]:
            can = (self.grid == car_type) & (np.roll(self.grid, roll_chk, axis=axis) == 0)
            if can.any():
                g = self.grid.copy()
                g[can] = 0
                g[np.roll(can, roll_dst, axis=axis)] = val
                self.grid = g


# colour map: 0=white, 1=royalblue (horizontal), 2=tomato (vertical)
cmap_bml = mcolors.ListedColormap(['white', 'royalblue', 'tomato'])
norm_bml = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap_bml.N)
legend_elems = [
    Patch(facecolor='royalblue', label='Horizontal \u2192'),
    Patch(facecolor='tomato',    label='Vertical \u2191'),
    Patch(facecolor='white', edgecolor='grey', label='Empty'),
]
print('Setup complete.')

## Animation 1: Three Density Regimes Side by Side

In [ ]:
# Estimate rho_c quickly
rhos_q = np.linspace(0.10, 0.55, 20)
flows_q = []
for i, rho in enumerate(rhos_q):
    s = BML(L=60, density=rho, seed=i)
    for _ in range(300): s.step()
    mf = []
    for _ in range(100):
        n = int((s.grid > 0).sum())
        mh = int(((s.grid==1)&(np.roll(s.grid,-1,axis=1)==0)).sum())
        mv = int(((s.grid==2)&(np.roll(s.grid, 1,axis=0)==0)).sum())
        s.step()
        mf.append((mh+mv)/max(n,1))
    flows_q.append(np.mean(mf))
dphi = -np.gradient(flows_q, rhos_q)
rho_c = float(rhos_q[np.argmax(dphi)])
print(f'\u03c1_c \u2248 {rho_c:.2f}')

scenarios = [
    (0.10, f'Free Flow  (\u03c1 = 0.10)'),
    (rho_c, f'Critical  (\u03c1 \u2248 {rho_c:.2f})'),
    (0.50, 'Gridlock  (\u03c1 = 0.50)'),
]

# initialise sims and warm up
sims = []
for rho, _ in scenarios:
    s = BML(L=L, density=rho, seed=7)
    for _ in range(T_WARMUP): s.step()
    sims.append(s)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
ims = []
for ax, (rho, title), sim in zip(axes, scenarios, sims):
    im = ax.imshow(sim.grid, cmap=cmap_bml, norm=norm_bml,
                   interpolation='nearest', animated=True)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ims.append(im)

axes[0].legend(handles=legend_elems, loc='lower right', fontsize=8)
step_text = fig.text(0.5, 0.01, 'Step 0', ha='center', fontsize=11)
fig.suptitle(f'BML Grid Evolution  (L = {L})', fontsize=13)
plt.tight_layout()

def update(frame):
    for im, sim in zip(ims, sims):
        sim.step()
        im.set_data(sim.grid)
    step_text.set_text(f'Step {T_WARMUP + frame + 1}')
    return ims + [step_text]

ani = animation.FuncAnimation(fig, update, frames=T_FRAMES,
                               interval=INTERVAL, blit=True)
plt.close()
HTML(ani.to_jshtml())

## Animation 2: Density Sweep — Watch the Transition

A single grid whose density slowly increases from $\rho = 0.10$ to $\rho = 0.55$.
Cars are added randomly at each density step. Watch for the moment the grid freezes.

In [ ]:
# Build a sequence of grids at increasing densities
rhos_sweep  = np.linspace(0.10, 0.55, T_FRAMES)
frames_sweep = []
rng_sw = np.random.default_rng(42)

for rho in rhos_sweep:
    s = BML(L=L, density=rho, seed=int(rho * 1000))
    for _ in range(300): s.step()
    frames_sweep.append(s.grid.copy())

fig2, ax2 = plt.subplots(figsize=(6, 6))
im2 = ax2.imshow(frames_sweep[0], cmap=cmap_bml, norm=norm_bml, interpolation='nearest')
ax2.legend(handles=legend_elems, loc='lower right', fontsize=9)
title2 = ax2.set_title(f'Density = {rhos_sweep[0]:.2f}', fontsize=12)
ax2.set_xlabel('Column')
ax2.set_ylabel('Row')
plt.tight_layout()

def update2(frame):
    im2.set_data(frames_sweep[frame])
    title2.set_text(f'Density \u03c1 = {rhos_sweep[frame]:.2f}')
    return [im2, title2]

ani2 = animation.FuncAnimation(fig2, update2, frames=T_FRAMES,
                                interval=120, blit=True)
plt.close()
HTML(ani2.to_jshtml())

## Animation 3: Space–Time Buildup

Instead of a grid snapshot, watch the **space–time diagram build up row by row**.
Each new row added to the bottom is the occupancy of row $L/2$ at the next timestep.
Black = any car present; white = empty.

Run at the critical density to see intermittent jamming patterns emerge over time.

In [ ]:
T_ST   = 150   # total timesteps to record
ROW    = L // 2

# Pre-record occupancy for three densities
st_data = {}
for rho, label in [(0.10, 'free'), (rho_c, 'critical'), (0.50, 'gridlock')]:
    s = BML(L=L, density=rho, seed=7)
    for _ in range(T_WARMUP): s.step()
    rows = []
    for _ in range(T_ST):
        rows.append((s.grid[ROW] > 0).astype(np.int8))
        s.step()
    st_data[label] = np.array(rows)

fig3, axes3 = plt.subplots(1, 3, figsize=(15, 4))
st_titles = [
    f'Free Flow  (\u03c1 = 0.10)',
    f'Critical  (\u03c1 \u2248 {rho_c:.2f})',
    'Gridlock  (\u03c1 = 0.50)',
]
labels_order = ['free', 'critical', 'gridlock']

canvas   = [np.zeros((T_ST, L), dtype=np.int8) for _ in range(3)]
im3_list = []
for ax, title, key in zip(axes3, st_titles, labels_order):
    im = ax.imshow(canvas[0], cmap='binary', aspect='auto',
                   interpolation='nearest', vmin=0, vmax=1, animated=True)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Column')
    ax.set_ylabel('Time (steps)')
    im3_list.append((im, key))

fig3.suptitle(f'Space\u2013Time Diagram Building Up  (row {ROW})', fontsize=13)
plt.tight_layout()

def update3(frame):
    artists = []
    for (im, key), cvs in zip(im3_list, canvas):
        cvs[frame] = st_data[key][frame]
        im.set_data(cvs)
        artists.append(im)
    return artists

ani3 = animation.FuncAnimation(fig3, update3, frames=T_ST,
                                interval=60, blit=True)
plt.close()
HTML(ani3.to_jshtml())